# ExaHyPE 2 Tutorials - Part 5: 3D Elastic Waves (and a Taste of HPC)

The final tutorial steps up to **three dimensions** and to a system with **nine unknowns**: the linear elastic wave equations, the workhorse of computational seismology. An isotropic *explosion* at the centre of a cube radiates an outgoing spherical wave, which we visualise by **slicing the 3D cube** interactively.

A 3D high-order ADER-DG solver is also a genuinely **HPC-scale** workload - lots of floating-point work per cell - so this part doubles as a tour of how ExaHyPE uses parallel hardware. At the end we measure its **MPI strong scaling** directly.

The implementation is in **`ADERDGSolver.cpp`**; open it alongside this notebook.

## The Elastic Wave Equations

We use the first-order **velocity-stress** form. The unknowns are the velocity $\mathbf{v} = (v_x, v_y, v_z)$ and the symmetric stress tensor $\boldsymbol{\sigma} = (\sigma_{xx}, \sigma_{yy}, \sigma_{zz}, \sigma_{xy}, \sigma_{xz}, \sigma_{yz})$ - nine fields in total:

$$
\rho\,\partial_t v_i = \partial_j \sigma_{ij}, \qquad
\partial_t \sigma_{ij} = \lambda\, \delta_{ij}\, \partial_k v_k
+ \mu\left(\partial_i v_j + \partial_j v_i\right).
$$

Here $\rho$ is the density and $\lambda, \mu$ are the Lamé parameters. The medium carries two wave types: fast **P-waves** at $c_p = \sqrt{(\lambda + 2\mu)/\rho}$ and slower **S-waves** at $c_s = \sqrt{\mu/\rho}$. We pick $\rho = 1$, $c_p = 2$, $c_s = 1$. An isotropic explosion (a Gaussian bump on the diagonal stresses) excites only P-waves, so we expect a clean expanding spherical shell.

## Importing the API

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../python/"))
import exahype2

## Defining the Project

A $3 \times 3 \times 3$ **cube** centred on the origin with non-periodic walls. The only change from the 2D tutorials is `dimensions = 3` and the three-component `size`/`offset`/`periodic_bc`.

In [ ]:
my_project = exahype2.Project(
    namespace    = ["tutorials", "exahype2", "elastic"],
    project_name = "Elastic",
    executable   = "Elastic",
    dimensions   = 3,
    size         = [ 3.0,  3.0,  3.0],
    offset       = [-1.5, -1.5, -1.5],
    periodic_bc  = [False, False, False],
)

## Defining the Solver

An ADER-DG solver of order 4. The unknowns are declared as a dictionary, so the velocity `v` has multiplicity 3 and the stress `sigma` multiplicity 6; we address them by name in C++ (`Shortcuts::v + 0..2`, `Shortcuts::sigma + 0..5`).

In [ ]:
aderdg_solver = exahype2.solvers.aderdg.GlobalAdaptiveTimeStep(
    name                 = "ADERDGSolver",
    order                = 4,
    time_step_relaxation = 0.9,
    unknowns             = {"v": 3, "sigma": 6},
    auxiliary_variables  = {},
)

We implement the **flux**, the **maximum eigenvalue** ($c_p$) and the initial and boundary conditions. The elastic system is linear, so we switch on the optimised linear ADER-DG kernels.

In [ ]:
aderdg_solver.set_implementation(
    initial_conditions  = exahype2.solvers.PDETerms.User_Defined_Implementation,
    boundary_conditions = exahype2.solvers.PDETerms.User_Defined_Implementation,
    flux                = exahype2.solvers.PDETerms.User_Defined_Implementation,
    max_eigenvalue      = exahype2.solvers.PDETerms.User_Defined_Implementation,
)

aderdg_solver.add_kernel_optimisations(is_linear=True)

my_project.add_solver(aderdg_solver)

## Building the Project

Generate and compile. In 3D at order 4 each cell carries $(4+1)^3 = 125$ degrees of freedom per unknown, so the build and the run are heavier than in the 2D tutorials.

In [ ]:
my_project.output.makefile.add_cpp_file("ADERDGSolver.cpp")
my_project.generate_and_build()

## The Implementation File

Open **`ADERDGSolver.cpp`**. `initialCondition` sets the Gaussian explosion (equal diagonal stresses, at rest); `boundaryConditions` gives a quiescent exterior to reflect off; `maxEigenvalue` returns the P-wave speed $c_p$; and `flux` is the velocity-stress flux, written out per direction with the Lamé parameters recovered from $\rho, c_p, c_s$.

## Running the Simulation

In [ ]:
import os, subprocess

# Runtime parameters - edit these freely, no rebuild needed.
end_time  = 0.5     # -et : physical end time
snapshots = 10      # -ns : number of output snapshots (drives the animation)
level     = 2       # -l  : uniform mesh refinement level (3^level cells per axis)

# Run with 4 OpenMP threads (the notebook equivalent of: export OMP_NUM_THREADS=4)
os.environ["OMP_NUM_THREADS"] = "4"

subprocess.run(
    ["./Elastic", "-et", str(end_time), "-ns", str(snapshots), "-l", str(level)],
    check=True,
)

## Visualising in 3D

A 3D field is best explored **slice by slice**. `show_solution_3d` cuts axis-aligned 2D planes through the cube: pick the slice axis with the **slice ⊥** dropdown (x/y/z), move the **slice** slider through the volume, pick a field, and animate with the **time** slider / **Play**. The first frame appears immediately; the viewer then briefly prefetches the remaining time frames of the current slice in parallel (a short *Prefetching N/M* wait), after which scrubbing and playback are perfectly smooth.

The expanding spherical P-wave appears in the mid-plane slice as **concentric rings** - the cross-section of the spherical shell. Move the slice slider towards a face of the cube and the rings shrink, exactly as cutting a sphere off-centre should.

*(The reference pictures in the documentation were rendered with `level = 3` - raise it in the run cell and re-run for the same crispness; the run then takes a few minutes.)*

In [ ]:
import importlib, os, sys
sys.path.insert(0, os.path.abspath("../../python/"))

from postprocessing import render_solution_interactively
importlib.reload(render_solution_interactively)

# Interactive slice viewer: pick the slice axis (x/y/z), move the slice slider
# through the cube, pick a field and animate with the time slider / Play.
render_solution_interactively.show_solution_3d("ADERDGSolver", field="sigma")

## Going Parallel - HPC in ExaHyPE 2

A 3D high-order ADER-DG solver is exactly the kind of compute-bound workload HPC exists for. ExaHyPE exposes parallelism through **two independent control surfaces**, neither of which needs a code change:

* **Distributed memory (MPI).** Launch under `mpirun -n N ./Elastic ...`: ExaHyPE decomposes the domain across the ranks and balances the load automatically. The startup banner reports `MPI   N Rank(s)` and the domain-decomposition `Load Balancing` statistics.
* **Shared memory (OpenMP).** The number of threads per rank is the `OMP_NUM_THREADS` environment variable; the banner echoes it as `OpenMP   N Thread(s)`.

Below we measure the first one: **MPI strong scaling**. We run the same problem on 1, 2 and 4 ranks - with one thread per rank, so the measurement is pure MPI - and read the `Average Time per Time Step` the solver prints, with plotting switched off (`-ns 0`) so we time pure compute.

In [ ]:
import os, re, subprocess

level = 2

# OpenMPI refuses to start as root (which the tutorial container is) without
# this confirmation; other MPI implementations simply ignore these variables.
os.environ["OMPI_ALLOW_RUN_AS_ROOT"] = "1"
os.environ["OMPI_ALLOW_RUN_AS_ROOT_CONFIRM"] = "1"

def time_per_step(ranks):
    """Run the solver on `ranks` MPI ranks; return the average time/step (s)."""
    env = dict(os.environ, OMP_NUM_THREADS="1")   # one thread per rank: pure MPI
    log = subprocess.run(
        ["mpirun", "-n", str(ranks), "./Elastic", "-et", "0.3", "-ns", "0", "-l", str(level)],
        env=env, capture_output=True, text=True, check=True,
    ).stdout
    timing = float(re.search(r"Average Time per Time Step\s+([\d.eE+-]+)\s*s", log).group(1))
    print(timing)
    return timing

rank_counts = [1, 2, 4]
timings = [time_per_step(n) for n in rank_counts]
for n, t in zip(rank_counts, timings):
    print(f"{n} rank(s): {t * 1e3:6.1f} ms/step   speedup x{timings[0] / t:.2f}")

Plotting the speedup against the rank count shows how close we get to the ideal linear line. The gap is the usual strong-scaling reality - halo exchange between the subdomains, load imbalance and the serial fraction of the traversal - and it widens as you add ranks while the problem size stays fixed.

In [ ]:
import matplotlib.pyplot as plt

speedup = [timings[0] / t for t in timings]
plt.figure(figsize=(5, 4))
plt.plot(rank_counts, speedup, "o-", label="measured")
plt.plot(rank_counts, rank_counts, "k--", label="ideal (linear)")
plt.xlabel("MPI ranks")
plt.ylabel("speedup")
plt.title("MPI strong scaling")
plt.legend()
plt.show()

## Exercise - Launch a Shear Wave

The isotropic explosion sets the three **diagonal** stresses equal. That is a pure
volume change, so it radiates only the fast **P-wave** - the clean spherical shell
(a ring in any slice). A **shear** source - a single *off-diagonal* stress -
instead launches the slower **S-wave** ($c_s = 1$, half the P-wave speed), whose
radiation is not spherical but a four-lobed **quadrupole**.

**Your task.** In `ADERDGSolver.cpp`, inside `initialCondition`, replace the three
diagonal-stress lines

```cpp
Q[Shortcuts::sigma + 0] = bump;   // sigma_xx
Q[Shortcuts::sigma + 1] = bump;   // sigma_yy
Q[Shortcuts::sigma + 2] = bump;   // sigma_zz
```

with a single off-diagonal (shear) component:

```cpp
Q[Shortcuts::sigma + 3] = bump;   // sigma_xy (shear) -> launches an S-wave
```

Then re-run the **build** cell and the **run** cell, and run the viewer below with
the slice axis on **z** at the middle slice position: in place of the P-wave's
expanding ring you will see the four lobes of the S-wave.

In [ ]:
# After editing the IC to a shear source and re-running build + run, view the
# velocity field sliced along z at the mid-plane: the four-lobed S-wave
# quadrupole replaces the P-wave's ring.
render_solution_interactively.show_solution_3d("ADERDGSolver", field="v")

## Where to Go Next

* **Push the scaling.** Raise `level` to 3 (a $27^3$-cell grid) for a much heavier run, then re-measure - bigger problems usually scale better. Add 8 to `rank_counts` if your machine has the cores.
* **Go hybrid.** Combine both control surfaces: give each rank several threads, e.g. `OMP_NUM_THREADS=4 mpirun -n 2 ./Elastic -et 0.3 -ns 0 -l 2` from a terminal, and watch both banner lines. On clusters this is the standard configuration - one rank per node or socket, threads inside.
* **Raise the order** (e.g. 5) for more accuracy and far more arithmetic per cell - and watch the per-step time and the DoF throughput climb.

That concludes the tutorial series - you have driven ADER-DG and Finite Volumes, adaptive and static mesh refinement, a-posteriori limiting, and a parallel 3D solver. Congratulations!